### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")


In [11]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000019D7FC8FE80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019D7FC8FD30>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [12]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [13]:
result

AIMessage(content='Bonjour, comment allez‑vous\u202f?', additional_kwargs={'reasoning_content': 'We need to translate "Hello How are you?" into French. Probably "Bonjour, comment ça va ?" or "Bonjour, comment allez-vous ?" The user says "Hello How are you?" There\'s no context. The best translation: "Bonjour, comment ça va ?" Or "Bonjour, comment allez-vous ?" The more casual: "Salut, ça va?" But "Hello How are you?" is formal greeting. So "Bonjour, comment allez-vous ?" Let\'s provide that.'}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 86, 'total_tokens': 198, 'completion_time': 0.116734257, 'completion_tokens_details': {'reasoning_tokens': 95}, 'prompt_time': 0.004720356, 'prompt_tokens_details': None, 'queue_time': 0.284514483, 'total_time': 0.121454613}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--01a1de13-4567-769d-b304-bb7b576de9d1-0'

In [14]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez‑vous\u202f?'

In [15]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment vas‑tu\u202f?'

In [16]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)


In [17]:
result=prompt.invoke({"language":"Hindi","text":"Hello"})

In [19]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into Hindi:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [21]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"Hindi","text":"Hello"})

'नमस्ते'